find('a')['href']

내가 쓴 최종 코드 

In [9]:
# riss.kr 에서 특정 키워드로 논문 / 학술 자료 검색하기

# Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
import time
import math
from bs4 import BeautifulSoup
import pandas as pd

# Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.")
print("=" *100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까? : ')

# Step 3. 수집된 데이터를 저장할 파일 이름 입력받기
ft_name = input('2.결과를 저장할 txt형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.txt): ')
fc_name = input('3.결과를 저장할 csv형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.csv): ')
fx_name = input('4.결과를 저장할 xlsx형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.xlsx): ')

# Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'https://www.riss.kr/'
driver.get(url)
time.sleep(3)
driver.maximize_window()

# Step 5. 검색어 입력 후 조회
element = driver.find_element(By.ID,'query')
element.click()
element.send_keys(query_txt)
element.send_keys("\n")

# Step 6. 학위논문 선택
driver.find_element(By.LINK_TEXT,'국내학술논문').click()
time.sleep(2)

# Step 7. BeautifulSoup 로 현재 페이지 파싱
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

# Step 8. 총 검색 건수 확인
total_cnt = soup_1.find('span','num').get_text()
print('키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt))

collect_cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
collect_page_cnt = math.ceil(collect_cnt / 10)

print('%s 건을 수집하기 위해 %s 페이지를 조회합니다.' %(collect_cnt,collect_page_cnt))
print('=' *80)

# Step 9. 데이터 저장용 리스트 생성
no2 = []
title2 = []
writer2 = []
org2 = []
year2 = []
thesis2 = []
url2 = []

no = 1

for a in range(1, collect_page_cnt + 1):

    html_2 = driver.page_source
    soup_2 = BeautifulSoup(html_2, 'html.parser')

    content_2 = soup_2.find('div','srchResultListW').find_all('li')

    for b in content_2:
        try:
            cont = b.find('div','cont')
            title_tag = cont.find('p','title').find('a')
            title = title_tag.get_text().strip()
        except:
            continue
        else:
            f = open(ft_name, 'a', encoding="UTF-8")

            # 번호
            print('1.번호:',no)
            no2.append(no)
            f.write('\n1.번호:' + str(no))

            # 제목
            print('2.논문제목:',title)
            title2.append(title)
            f.write('\n2.논문제목:' + title)

            # 저자
            writer = cont.find('span','writer').get_text().strip()
            print('3.저자:',writer)
            writer2.append(writer)
            f.write('\n3.저자:' + writer)

            # 소속기관
            org = cont.find('span','assigned').get_text().strip()
            print('4.소속기관:',org)
            org2.append(org)
            f.write('\n4.소속기관:' + org)
            etc = cont.find('p','etc')

            # 발행년도
            year = ""
            if etc:
                for s in etc.find_all('span'):
                    text = s.get_text().strip()
                    if text.isdigit():
                        year = text
                        break

            print("5.발행년도:", year)
            year2.append(year)
            f.write('\n5.발행년도:' + year)

            # 논문집 
            try:
                # etc 클래스 내부의 모든 span을 찾음
                etc_info = cont.find('p', 'etc').find_all('span')
                # 4번째 span에 위치한 학술지명 텍스트 추출
                thesis = etc_info[3].get_text(strip=True)
            except:
                thesis = "학술지명 미상"

            print('6.논문집:', thesis)
            thesis2.append(thesis)
            f.write('\n6.논문집: ' + thesis)
            
            # URL
            detail_url = cont.find('p','title').find('a')['href']
            print('7.URL:',detail_url)
            url2.append(detail_url)
            f.write('\n7.URL:' + detail_url + '\n')

            f.close()

            no += 1
            print("\n")

            if no > collect_cnt:
                break

            time.sleep(1)

    if no > collect_cnt:
        break

    # 페이지 이동
    next_page = a + 1
    time.sleep(1)

    try:
        driver.find_element(By.LINK_TEXT, str(next_page)).click()
    except:
        try:
            driver.find_element(By.LINK_TEXT,('다음 페이지로')).click()
        except:
            print("마지막 페이지입니다.")
            break

    time.sleep(2)

# Step 10. DataFrame 생성 및 저장
df = pd.DataFrame()
df['번호'] = no2
df['제목'] = title2
df['저자'] = writer2
df['소속(발행)기관'] = org2
df['발행년도'] = year2
df['논문집'] =  thesis2
df['URL'] = url2

# 엑셀 저장
df.to_excel(fx_name, index=False, engine='openpyxl')

# CSV 저장
df.to_csv(fc_name, index=False, encoding="utf-8-sig")

print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')
driver.quit()

 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.
키워드 해양자원 (으)로 총 1 건의 학위논문이 검색되었습니다
15 건을 수집하기 위해 2 페이지를 조회합니다.
1.번호: 1
2.논문제목: 발표논문 / 해양생물자원으로서 해조류 : 생물활성물질의 정제와 분자적 응용
3.저자: 홍용기(Yong Ki Hong)
4.소속기관: 한국조류학회
5.발행년도: 2000
6.논문집: 국제심포지움 일정 및 발표논문집 - 21세기,해양환경과해양생물자원의 전망
7.URL: /search/detail/DetailView.do?p_mat_type=1a0202e37d52c72d&control_no=508c47ca1906d5ecffe0bdc3ef48d419&keyword=해양자원


1.번호: 2
2.논문제목: 수거된 해양폐기물 자원화 기술 개발(Ⅰ)
3.저자: 길상인(Sang-In Keel),윤진한(Jin-Han Yun),최연석(Yeon-Seok Choi),강창구(Chang-Gu Kang),유정석(Jeong-Seok Yu)
4.소속기관: 한국해양환경·에너지학회
5.발행년도: 2002
6.논문집: 한국해양환경·에너지학회지
7.URL: /search/detail/DetailView.do?p_mat_type=1a0202e37d52c72d&control_no=09ec1e3541fcc118ffe0bdc3ef48d419&keyword=해양자원


1.번호: 3
2.논문제목: 조건부 가치측정법을 이용한 공해상 해양생명자원 확보의 경제적 가치 추정
3.저자: 진세준,권영주,최은철
4.소속기관: 해양환경안전학회
5.발행년도: 2023
6.논문집: 해양환경안전학회지
7.URL: /search/detail/DetailView.do?p_mat_type=1a0202e37d52c72d&control_no=59e951e287ce111647de9c1710b0298d&keyword=해양자원


1.번호: 4
2.논문제목: 해양수산생명자원의 국내 접근 절차 관련 법제도의 문제점 및 개선방향 :

진우님 코드 

In [8]:
# riss.kr 에서 특정 키워드로 논문 / 학술 자료 검색하기

# Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
import time
import math
from bs4 import BeautifulSoup
import pandas as pd

# Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.")
print("=" *100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까? : ')

# Step 3. 수집된 데이터를 저장할 파일 이름 입력받기
ft_name = input('2.결과를 저장할 txt형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.txt): ')
fc_name = input('3.결과를 저장할 csv형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.csv): ')
fx_name = input('4.결과를 저장할 xlsx형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.xlsx): ')

# Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'https://www.riss.kr/'
driver.get(url)
time.sleep(3)
driver.maximize_window()

# Step 5. 검색어 입력 후 조회
element = driver.find_element(By.ID,'query')
element.click()
element.send_keys(query_txt)
element.send_keys("\n")

# Step 6. 학위논문 선택
driver.find_element(By.LINK_TEXT,'국내학술논문').click()
time.sleep(2)

# Step 7. BeautifulSoup 로 현재 페이지 파싱
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

# Step 8. 총 검색 건수 확인
total_cnt = soup_1.find('span','num').get_text()
print('키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt))

collect_cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
collect_page_cnt = math.ceil(collect_cnt / 10)

print('%s 건을 수집하기 위해 %s 페이지를 조회합니다.' %(collect_cnt,collect_page_cnt))
print('=' *80)

# Step 9. 데이터 저장용 리스트 생성
no2 = [ ] #번호 저장
title2 = [ ] #논문제목 저장
writer2 = [ ] #논문저자 저장
org2 = [ ] #소속기관 저장
year2= [ ]
zip2=[ ]
url_t2= [ ]
no = 1


for a in range(1, collect_page_cnt + 1) :

    html_2 = driver.page_source
    soup_2 = BeautifulSoup(html_2, 'html.parser')

    content_2 = soup_2.find('div','srchResultListW').find_all('li')

    for b in content_2 :
        #1. 논문제목 있을 경우만
        try :
            title = b.find('div','cont').find('p','title').get_text()
        except :
            continue
        else :
            f = open(ft_name, 'a' , encoding="UTF-8")
            print('1.번호:',no)
            no2.append(no)
            f.write('\n'+'1.번호:' + str(no))

            print('2.논문제목:',title)
            title2.append(title)
            f.write('\n' + '2.논문제목:' + title)

            writer = b.find('span','writer').get_text()
            print('3.저자:',writer)
            writer2.append(writer)
            f.write('\n' + '3.저자:' + writer)

            org = b.find('span','assigned').get_text()
            print('4.소속기관:' , org)
            org2.append(org)
            f.write('\n' + '4.소속기관:' + org )

            spans = b.find_all('span')

            year_text = None
            for i,s in  enumerate(spans):
                text = s.get_text().strip()
                if text.isdigit() and len(text) == 4:
                    year_text = text
                    
                    if i + 1 < len(spans):
                        zip = spans[i + 1].get_text().strip()
                    break
                    
            print('5.발표년도:', year_text)
            year2.append(year_text)
            f.write('\n5.발표년도: ' + str(year_text))
        

            print('6.논문집', zip)
            zip2.append(zip)
            f.write('\n' + '6.논문집' + zip)

            url_t = b.find('a')['href']
            print('7.논문 url 정보:', url_t)
            url_t2.append(url_t)
            f.write('\n' + '.논문url정보' + url_t)
            

            f.close( )

            no += 1
            print("\n")

            if no > collect_cnt :
                break

            time.sleep(1) # 페이지 변경 전 1초 대기

    a += 1
    b = str(a)

    try :
        driver.find_element(By.LINK_TEXT ,'%s' %b).click()
        time.sleep(1)
    except :
        driver.find_element(By.CLASS_NAME, 'next1').click()
        time.sleep(1)
print("요청하신 작업이 모두 완료되었습니다")





# Step 10. DataFrame 생성 및 저장
df = pd.DataFrame()
df['번호'] = no2
df['제목'] = title2
df['저자'] = writer2
df['소속(발행)기관'] = org2
df['발행년도'] = year2
df['논문집'] =  zip2
df['URL'] = url_t2

# 엑셀 저장
df.to_excel(fx_name, index=False, engine='openpyxl')

# CSV 저장
df.to_csv(fc_name, index=False, encoding="utf-8-sig")

print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')
driver.quit()

 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.
키워드 해양자원 (으)로 총 1 건의 학위논문이 검색되었습니다
15 건을 수집하기 위해 2 페이지를 조회합니다.
1.번호: 1
2.논문제목: 발표논문 / 해양생물자원으로서 해조류 : 생물활성물질의 정제와 분자적 응용
3.저자: 홍용기(Yong Ki Hong)
4.소속기관: 한국조류학회
5.발표년도: 2000
6.논문집 국제심포지움 일정 및 발표논문집 - 21세기, 해양환경과 해양생물자원의 전망
7.논문 url 정보: /search/detail/DetailView.do?p_mat_type=1a0202e37d52c72d&control_no=508c47ca1906d5ecffe0bdc3ef48d419&keyword=해양자원


1.번호: 2
2.논문제목: 수거된 해양폐기물 자원화 기술 개발(Ⅰ)
3.저자: 길상인(Sang-In Keel),윤진한(Jin-Han Yun),최연석(Yeon-Seok Choi),강창구(Chang-Gu Kang),유정석(Jeong-Seok Yu)
4.소속기관: 한국해양환경·에너지학회
5.발표년도: 2002
6.논문집 한국해양환경·에너지학회지
7.논문 url 정보: /search/detail/DetailView.do?p_mat_type=1a0202e37d52c72d&control_no=09ec1e3541fcc118ffe0bdc3ef48d419&keyword=해양자원


1.번호: 3
2.논문제목: 조건부 가치측정법을 이용한 공해상 해양생명자원 확보의 경제적 가치 추정
3.저자: 진세준,권영주,최은철
4.소속기관: 해양환경안전학회
5.발표년도: 2023
6.논문집 해양환경안전학회지
7.논문 url 정보: /search/detail/DetailView.do?p_mat_type=1a0202e37d52c72d&control_no=59e951e287ce111647de9c1710b0298d&keyword=해양자원


1.번호: 4
2.논문제목: 해양수산생명자원의 국내 접근 절차 관련 

In [ ]:
#!pip install pandas

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 9.7/9.7 MB 56.8 MB/s  0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   -------------------------------- ------- 10.0/12.3 MB 63.9 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 43.5 MB/s  0:00:00

   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   -----------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#시행착오


# riss.kr 에서 특정 키워드로 논문 / 학술 자료 검색하기

# Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
import time
import math
from bs4 import BeautifulSoup
import pandas as pd

# Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.")
print("=" *100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까? : ')

# Step 3. 수집된 데이터를 저장할 파일 이름 입력받기
ft_name = input('2.결과를 저장할 txt형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.txt): ')
fc_name = input('3.결과를 저장할 csv형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.csv): ')
fx_name = input('4.결과를 저장할 xlsx형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.xlsx): ')

# Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'https://www.riss.kr/'
driver.get(url)
time.sleep(3)
driver.maximize_window()

# Step 5. 검색어 입력 후 조회
element = driver.find_element(By.ID,'query')
element.click()
element.send_keys(query_txt)
element.send_keys("\n")

# Step 6. 학위논문 선택
driver.find_element(By.LINK_TEXT,'국내학술논문').click()
time.sleep(2)

# Step 7. BeautifulSoup 로 현재 페이지 파싱
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

# Step 8. 총 검색 건수 확인
total_cnt = soup_1.find('span','num').get_text()
print('키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt))

collect_cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
collect_page_cnt = math.ceil(collect_cnt / 10)

print('%s 건을 수집하기 위해 %s 페이지를 조회합니다.' %(collect_cnt,collect_page_cnt))
print('=' *80)

# Step 9. 데이터 저장용 리스트 생성
no2 = []
title2 = []
writer2 = []
org2 = []
year2 = []
thesis2 = []
url2 = []

no = 1

for a in range(1, collect_page_cnt + 1):

    html_2 = driver.page_source
    soup_2 = BeautifulSoup(html_2, 'html.parser')

    content_2 = soup_2.find('div','srchResultListW').find_all('li')

    for b in content_2:
        try:
            cont = b.find('div','cont')
            title_tag = cont.find('p','title').find('a')
            title = title_tag.get_text().strip()
        except:
            continue
        else:
            f = open(ft_name, 'a', encoding="UTF-8")

            # 번호
            print('1.번호:',no)
            no2.append(no)
            f.write('\n1.번호:' + str(no))

            # 제목
            print('2.논문제목:',title)
            title2.append(title)
            f.write('\n2.논문제목:' + title)

            # 저자
            writer = cont.find('span','writer').get_text().strip()
            print('3.저자:',writer)
            writer2.append(writer)
            f.write('\n3.저자:' + writer)

            # 소속기관
            org = cont.find('span','assigned').get_text().strip()
            print('4.소속기관:',org)
            org2.append(org)
            f.write('\n4.소속기관:' + org)

            # 발행년도 
            year =cont.find('span','assigned').find('span').get_text().strip()
            print('5.발행년도:', year)
            year2.append(year)
            f.write('\n5.발행년도:'+year)

            # 논문집 
            thesis =cont.find('etc','highlight').get_text().strip()
            print('6.논문집:',thesis )
            thesis2.append(thesis)
            f.write('\6.논문집:', thesis)
            
            # URL
            detail_url = cont.find('a')['href'].get_text().strip()
            print('7.URL:',detail_url)
            url2.append(detail_url)
            f.write('\n7.URL:' + detail_url + '\n')

            f.close()

            no += 1
            print("\n")

            if no > collect_cnt:
                break

            time.sleep(1)

    if no > collect_cnt:
        break

    # 페이지 이동
    next_page = a + 1
    time.sleep(1)

    try:
        driver.find_element(By.LINK_TEXT, str(next_page)).click()
    except:
        try:
            driver.find_element(By.LINK_TEXT,('다음 페이지로')).click()
        except:
            print("마지막 페이지입니다.")
            break

    time.sleep(2)

# Step 10. DataFrame 생성 및 저장
df = pd.DataFrame()
df['번호'] = no2
df['제목'] = title2
df['저자'] = writer2
df['소속(발행)기관'] = org2
df['발행년도'] = year2
df['논문집'] = degree2
df['URL'] = url2

# 엑셀 저장
df.to_excel(fx_name, index=False, engine='openpyxl')

# CSV 저장
df.to_csv(fc_name, index=False, encoding="utf-8-sig")

print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')
driver.quit()

In [ ]:
# 시행착오 

# riss.kr 에서 특정 키워드로 논문 / 학술 자료 검색하기

# Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
import time
import math
from bs4 import BeautifulSoup
import pandas as pd

# Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.")
print("=" *100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까? : ')

# Step 3. 수집된 데이터를 저장할 파일 이름 입력받기
ft_name = input('2.결과를 저장할 txt형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.txt): ')
fc_name = input('3.결과를 저장할 csv형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.csv): ')
fx_name = input('4.결과를 저장할 xlsx형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.xlsx): ')

# Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'https://www.riss.kr/'
driver.get(url)
time.sleep(3)
driver.maximize_window()

# Step 5. 검색어 입력 후 조회
element = driver.find_element(By.ID,'query')
element.click()
element.send_keys(query_txt)
element.send_keys("\n")

# Step 6. 학위논문 선택
driver.find_element(By.LINK_TEXT,'국내학술논문').click()
time.sleep(2)

# Step 7. BeautifulSoup 로 현재 페이지 파싱
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

# Step 8. 총 검색 건수 확인
total_cnt = soup_1.find('span','num').get_text()
print('키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt))

collect_cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
collect_page_cnt = math.ceil(collect_cnt / 10)

print('%s 건을 수집하기 위해 %s 페이지를 조회합니다.' %(collect_cnt,collect_page_cnt))
print('=' *80)

# Step 9. 데이터 저장용 리스트 생성
no2 = []
title2 = []
writer2 = []
org2 = []
year2 = []
thesis2 = []
url2 = []

no = 1

for a in range(1, collect_page_cnt + 1):

    html_2 = driver.page_source
    soup_2 = BeautifulSoup(html_2, 'html.parser')

    content_2 = soup_2.find('div','srchResultListW').find_all('li')

    for b in content_2:
        try:
            cont = b.find('div','cont')
            title_tag = cont.find('p','title').find('a')
            title = title_tag.get_text().strip()
        except:
            continue
        else:
            f = open(ft_name, 'a', encoding="UTF-8")

            # 번호
            print('1.번호:',no)
            no2.append(no)
            f.write('\n1.번호:' + str(no))

            # 제목
            print('2.논문제목:',title)
            title2.append(title)
            f.write('\n2.논문제목:' + title)

            # 저자
            writer = cont.find('span','writer').get_text().strip()
            print('3.저자:',writer)
            writer2.append(writer)
            f.write('\n3.저자:' + writer)

            # 소속기관
            org = cont.find('span','assigned').get_text().strip()
            print('4.소속기관:',org)
            org2.append(org)
            f.write('\n4.소속기관:' + org)

            # 발행년도 
            etc = cont.find('p','etc')
            span_list = etc.find_all('span')

            year = span_list[2].get_text().strip()
            print("5.발행년도:", year)

            # 논문집 
            etc = cont.find('p','etc')

            journal = etc.find('a').get_text(strip=True)
            print("6.논문집:", journal)
            
            # URL
            detail_url = cont.find('p','title').find('a')['href']
            print('7.URL:',detail_url)
            url2.append(detail_url)
            f.write('\n7.URL:' + detail_url + '\n')

            f.close()

            no += 1
            print("\n")

            if no > collect_cnt:
                break

            time.sleep(1)

    if no > collect_cnt:
        break

    # 페이지 이동
    next_page = a + 1
    time.sleep(1)

    try:
        driver.find_element(By.LINK_TEXT, str(next_page)).click()
    except:
        try:
            driver.find_element(By.LINK_TEXT,('다음 페이지로')).click()
        except:
            print("마지막 페이지입니다.")
            break

    time.sleep(2)

# Step 10. DataFrame 생성 및 저장
df = pd.DataFrame()
df['번호'] = no2
df['제목'] = title2
df['저자'] = writer2
df['소속(발행)기관'] = org2
df['발행년도'] = year2
df['논문집'] =  thesis2
df['URL'] = url2

# 엑셀 저장
df.to_excel(fx_name, index=False, engine='openpyxl')

# CSV 저장
df.to_csv(fc_name, index=False, encoding="utf-8-sig")

print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')
driver.quit()